In [1]:
# faiss installation
!pip install faiss-cpu




[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# You can also use the below command, although, both of them are same.
%pip install faiss-cpu


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import faiss

In [4]:
faiss.__version__

'1.14.3'

In [5]:
text_data = """
Meeting started around 9:15 AM although a few people joined later. Main discussion was about the migration timeline. Nobody seemed completely certain whether Phase 2 should begin next Monday or after the quarterly review. Need confirmation from the infrastructure team.

Reminder:

Check backup logs.
Verify user permissions.
Send updated spreadsheet.
Maybe ask finance if budget approval has actually been received?

Yesterday felt unusually busy. Emails, calls, more emails... somewhere in between there was a request to generate three reports but only two were completed before lunch. The third one? Still pending because the source data looked incomplete. Strange thing was, the totals matched but individual records didn't.

Reference IDs:
INV-20481
REQ-9982
PO# 77124-B
Status = Pending

Weather forecast says 32°C tomorrow. Not sure that's accurate because it was supposed to rain today as well. Instead, sunshine all afternoon. By 5 PM the clouds finally appeared, then disappeared again within twenty minutes.

Coffee machine — still broken.

Random observations...

The office printer makes more noise than necessary.

Someone left a notebook in Conference Room B.

Password policy updated? I think so, but haven't read the announcement completely.

Need to remember:
database cleanup,
archive old logs,
delete duplicate files,
check API timeout values (currently 30? maybe 60?)

The documentation isn't exactly wrong, just... incomplete. Section 4 mentions configuration files but skips the environment variables entirely. Then Section 6 suddenly assumes everything has already been deployed. Took almost forty minutes to figure out what was missing.

Error messages seen today:

Connection timeout.
Invalid token.
File not found.
Unexpected NULL value.

Shopping list

milk

bread

charger (USB-C?)

HDMI cable

AA batteries

Don't forget birthday gift. Date... 18th? Need to verify.

Sometimes projects look simple until implementation begins. One requirement becomes five, five become twenty. Documentation gets updated, then updated again. Halfway through someone remembers another dependency that nobody discussed earlier. Eventually everything works, but not exactly the way it was first imagined.

11:48 PM

Still debugging.

Function returns correct output for 98% of records.

Remaining 2%...
No obvious pattern.
Could be encoding?
Could be whitespace?
Could be both.

Need another look tomorrow.
"""


In [6]:
import requests
import os
import json
import numpy as np

The above textual data is big. So, rather than embedding all the data in a single array or list, we are first chunking the whole textual data. So, instead of processing the whole data together, we divide it into smaller pieces called as chunking.

Imagine a book.
```
Book
---------------------------------------------------
Page1
Page2
Page3
Page4
Page5
...
---------------------------------------------------

Instead of giving the whole book to an LLM, we divide it into smaller parts.

Chunk 1
Chunk 2
Chunk 3
Chunk 4
```
That's exactly what the below function does.

#### What do chunk_size and overlap actually mean?
Think of it with a simple example. Say you have 15 words:
```
[W1, W2, W3, W4, W5, W6, W7, W8, W9, W10, W11, W12, W13, W14, W15]
```
With chunk_size=6 and overlap=2, the step size is 6 - 2 = 4. That means each chunk grabs 6 words, but the starting position moves forward by only 4, so the last 2 words of one chunk reappear at the start of the next:

```
Chunk 1: [W1,  W2,  W3,  W4,  W5,  W6 ]    ← start=0, end=6
Chunk 2:              [W5,  W6,  W7,  W8,  W9,  W10]    ← start=4, end=10
Chunk 3:                          [W9,  W10, W11, W12, W13, W14]    ← start=8, end=14
Chunk 4:                                      [W13, W14, W15]    ← start=12, end=15
```

Notice how W5-W6 appear in both Chunk 1 and Chunk 2, and W9-W10 appear in both Chunk 2 and Chunk 3. That's the overlap in action. It ensures that context at chunk boundaries isn't lost, which is especially important if these chunks are being fed to an LLM or a search system.

The **key rule:** 
- overlap must always < chunk_size. Otherwise, the step size becomes zero or negative, and the function breaks.

In [7]:
def chunk_text(text, chunk_size=200, overlap=50):
    if overlap >= chunk_size:
        raise ValueError(
            f"overlap ({overlap}) must be less than chunk_size ({chunk_size})"
        )

    words = text.split()
    chunks = []
    # core logic of the function starts here which deteremines where each chunk starts 
    # and ends and ensures that the overlap is maintained between chunks
    start = 0
    step = chunk_size - overlap

    while start < len(words): # Continue creating chunks until we've processed all words
        end = min(start + chunk_size, len(words)) # calculates where the chunk should end
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += step

    return chunks

In [8]:
chunks = chunk_text(text_data)
chunks

["Meeting started around 9:15 AM although a few people joined later. Main discussion was about the migration timeline. Nobody seemed completely certain whether Phase 2 should begin next Monday or after the quarterly review. Need confirmation from the infrastructure team. Reminder: Check backup logs. Verify user permissions. Send updated spreadsheet. Maybe ask finance if budget approval has actually been received? Yesterday felt unusually busy. Emails, calls, more emails... somewhere in between there was a request to generate three reports but only two were completed before lunch. The third one? Still pending because the source data looked incomplete. Strange thing was, the totals matched but individual records didn't. Reference IDs: INV-20481 REQ-9982 PO# 77124-B Status = Pending Weather forecast says 32°C tomorrow. Not sure that's accurate because it was supposed to rain today as well. Instead, sunshine all afternoon. By 5 PM the clouds finally appeared, then disappeared again within 

In [9]:
url = "https://api.euron.one/api/v1/euri/embeddings"

header = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {os.environ.get('EURI_API_KEY', 'your-api-key-here')}"
}

model_name = "text-embedding-3-small"

In [14]:
all_embeddings = [] # Initialize an empty list to store embeddings for all chunks
for i, chunk in enumerate(chunks):
    payload = {
        "model": model_name,
        "input": chunk
    }

    request_response = requests.post(url, headers=header, data=json.dumps(payload))
    data = request_response.json()
    embeddings = data['data'][0]['embedding']
    all_embeddings.append(embeddings) 

In [16]:
all_embeddings

[[0.0018405914306640625,
  0.05908203125,
  0.0287628173828125,
  -0.06597900390625,
  0.004871368408203125,
  0.042694091796875,
  0.0015087127685546875,
  0.03094482421875,
  0.0094757080078125,
  -0.0158233642578125,
  0.006000518798828125,
  0.01306915283203125,
  -0.01739501953125,
  0.01137542724609375,
  -0.025787353515625,
  0.03887939453125,
  -0.00414276123046875,
  0.004421234130859375,
  -0.047332763671875,
  0.04010009765625,
  0.039093017578125,
  0.0100860595703125,
  -0.02349853515625,
  0.00616455078125,
  -0.081787109375,
  0.01357269287109375,
  -0.0298004150390625,
  0.0242462158203125,
  0.0260772705078125,
  -0.06939697265625,
  0.00467681884765625,
  -0.025299072265625,
  -0.0399169921875,
  0.03533935546875,
  -0.0247344970703125,
  0.019256591796875,
  0.000461578369140625,
  0.00669097900390625,
  -0.01690673828125,
  -0.017822265625,
  -0.0279388427734375,
  -0.0121002197265625,
  -0.0151519775390625,
  -0.0021152496337890625,
  -0.01076507568359375,
  -0.017

In [19]:
all_embeddings[0]

[0.0018405914306640625,
 0.05908203125,
 0.0287628173828125,
 -0.06597900390625,
 0.004871368408203125,
 0.042694091796875,
 0.0015087127685546875,
 0.03094482421875,
 0.0094757080078125,
 -0.0158233642578125,
 0.006000518798828125,
 0.01306915283203125,
 -0.01739501953125,
 0.01137542724609375,
 -0.025787353515625,
 0.03887939453125,
 -0.00414276123046875,
 0.004421234130859375,
 -0.047332763671875,
 0.04010009765625,
 0.039093017578125,
 0.0100860595703125,
 -0.02349853515625,
 0.00616455078125,
 -0.081787109375,
 0.01357269287109375,
 -0.0298004150390625,
 0.0242462158203125,
 0.0260772705078125,
 -0.06939697265625,
 0.00467681884765625,
 -0.025299072265625,
 -0.0399169921875,
 0.03533935546875,
 -0.0247344970703125,
 0.019256591796875,
 0.000461578369140625,
 0.00669097900390625,
 -0.01690673828125,
 -0.017822265625,
 -0.0279388427734375,
 -0.0121002197265625,
 -0.0151519775390625,
 -0.0021152496337890625,
 -0.01076507568359375,
 -0.0177154541015625,
 0.0038318634033203125,
 0.0123

In [21]:
len(all_embeddings)

3

In [22]:
len(chunks)

3

In [24]:
all_embeddings[1]

[-0.042144775390625,
 0.04510498046875,
 0.0218963623046875,
 -0.061187744140625,
 0.002819061279296875,
 0.01849365234375,
 0.0311279296875,
 0.005779266357421875,
 -0.0184478759765625,
 -0.0117034912109375,
 0.0296630859375,
 -0.010772705078125,
 -0.0352783203125,
 -0.0136566162109375,
 -0.0024890899658203125,
 0.03582763671875,
 -0.0244903564453125,
 0.0248870849609375,
 -0.045135498046875,
 0.035430908203125,
 0.052947998046875,
 0.0029277801513671875,
 0.0262908935546875,
 0.038421630859375,
 -0.035064697265625,
 -0.0035495758056640625,
 -0.03204345703125,
 0.036773681640625,
 0.06060791015625,
 -0.07330322265625,
 -0.00994110107421875,
 -0.026275634765625,
 -0.038482666015625,
 -0.00969696044921875,
 -0.0028209686279296875,
 0.0273284912109375,
 0.030548095703125,
 -0.031951904296875,
 -0.00397491455078125,
 -0.00778961181640625,
 -0.0109100341796875,
 -0.0158233642578125,
 -0.0188140869140625,
 -0.050323486328125,
 0.01123046875,
 0.00907135009765625,
 -0.0270233154296875,
 0.03

In [25]:
all_embeddings[2]

[-0.007320404052734375,
 0.028533935546875,
 0.051971435546875,
 -0.018646240234375,
 0.001953125,
 0.04510498046875,
 0.07574462890625,
 0.00274658203125,
 0.0126190185546875,
 0.0009975433349609375,
 0.038726806640625,
 -0.006885528564453125,
 -0.035064697265625,
 -0.0008234977722167969,
 -0.02642822265625,
 0.017303466796875,
 0.0294952392578125,
 0.004764556884765625,
 -0.0243682861328125,
 0.031707763671875,
 0.06707763671875,
 -0.064697265625,
 0.0247802734375,
 0.0099945068359375,
 0.01216888427734375,
 -0.020721435546875,
 -0.043914794921875,
 0.051025390625,
 0.038848876953125,
 -0.060150146484375,
 -0.040252685546875,
 -0.03448486328125,
 -0.00753021240234375,
 -0.0308380126953125,
 0.01446533203125,
 0.023193359375,
 0.050872802734375,
 -0.0036220550537109375,
 -0.0097198486328125,
 0.00467681884765625,
 -0.01071929931640625,
 0.001712799072265625,
 -0.047027587890625,
 -0.0189056396484375,
 0.0268402099609375,
 0.0297698974609375,
 -0.014190673828125,
 -0.005405426025390625

In [ ]:
all_embeddings[3] # we have checked the length of all_embeddings and it is 3, hence index out of range

IndexError: list index out of range

In [ ]:
type(all_embeddings)

list

**Important:**
- you cannot store a list directly inside FAISS, hence you need to convert it into float32 which is the native datatype of FAISS.
- float32 (32-bit single-precision floating-point) is the native, default data type used by FAISS for all input vectors and internal metric calculations. Even if your data uses other types, it must generally be cast to float32 before being appended to or queried against a CPU-based FAISS index. Since the data type of all_embeddings is 'list', we need to convert it into float32 using numpy.

In [28]:
embeddings_array = np.array(all_embeddings, dtype='float32')

In [ ]:
embeddings_array # NumPy array containing vector embeddings

array([[ 0.00184059,  0.05908203,  0.02876282, ..., -0.01293182,
         0.02587891, -0.02593994],
       [-0.04214478,  0.04510498,  0.02189636, ..., -0.0114212 ,
        -0.01802063,  0.00308037],
       [-0.0073204 ,  0.02853394,  0.05197144, ..., -0.02001953,
         0.00426865,  0.01495361]], shape=(3, 1536), dtype=float32)

**Index** in FAISS -> An index in FAISS is the core data structure that stores your vector embeddings and executes the mathematical searches to find the most similar vectors. Think of it like a database table, but instead of searching by text or IDs, it searches by geometric distance in high-dimensional space.

In [ ]:
# Instantiating a FAISS Index
faiss.IndexFlatL2(embeddings_array.shape[1]) # creates a FAISS index that performs exact nearest neighbor search using the L2 (Euclidean) distance metric

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000001B80FECF6B0> >

In [ ]:
# Create a FAISS index for exact nearest-neighbor search using L2 (Euclidean) distance, expecting 1536-dimensional embedding vectors
base_index = faiss.IndexFlatL2(1536)

In [ ]:
# adds all the embedding vectors stored in embeddings_array to the FAISS index, making them searchabl
base_index.add(embeddings_array)

In [33]:
faiss.write_index(base_index, "faiss_index.faiss") # saves the FAISS index to a file named "faiss_index.index" for later use

In [34]:
query_test = "Check backup logs and verify user permissions."

In [40]:
def embedding_text(text):
    payload = {
        "model": model_name,
        "input": chunk
    }

    request_response = requests.post(url, headers=header, data=json.dumps(payload))
    data = request_response.json()
    embeddings = data['data'][0]['embedding']
    emb = np.array(embeddings, dtype='float32').reshape(1, -1)
    return emb

In [41]:
query_test_emb = embedding_text(query_test)

In [42]:
query_test_emb

array([[-0.00733948,  0.02851868,  0.05197144, ..., -0.02003479,
         0.00426483,  0.01494598]], shape=(1, 1536), dtype=float32)

In [43]:
base_index.search(query_test_emb, 3)

(array([[7.0714952e-07, 6.6024959e-01, 1.0534058e+00]], dtype=float32),
 array([[2, 1, 0]]))

In [44]:
chunks

["Meeting started around 9:15 AM although a few people joined later. Main discussion was about the migration timeline. Nobody seemed completely certain whether Phase 2 should begin next Monday or after the quarterly review. Need confirmation from the infrastructure team. Reminder: Check backup logs. Verify user permissions. Send updated spreadsheet. Maybe ask finance if budget approval has actually been received? Yesterday felt unusually busy. Emails, calls, more emails... somewhere in between there was a request to generate three reports but only two were completed before lunch. The third one? Still pending because the source data looked incomplete. Strange thing was, the totals matched but individual records didn't. Reference IDs: INV-20481 REQ-9982 PO# 77124-B Status = Pending Weather forecast says 32°C tomorrow. Not sure that's accurate because it was supposed to rain today as well. Instead, sunshine all afternoon. By 5 PM the clouds finally appeared, then disappeared again within 

In [ ]:
chunks[2] # First nearest neighbor chunk retrieved from the FAISS index based on the query embedding

'another dependency that nobody discussed earlier. Eventually everything works, but not exactly the way it was first imagined. 11:48 PM Still debugging. Function returns correct output for 98% of records. Remaining 2%... No obvious pattern. Could be encoding? Could be whitespace? Could be both. Need another look tomorrow.'

In [49]:
chunks[1] # second nearest neighbor

"minutes. Coffee machine — still broken. Random observations... The office printer makes more noise than necessary. Someone left a notebook in Conference Room B. Password policy updated? I think so, but haven't read the announcement completely. Need to remember: database cleanup, archive old logs, delete duplicate files, check API timeout values (currently 30? maybe 60?) The documentation isn't exactly wrong, just... incomplete. Section 4 mentions configuration files but skips the environment variables entirely. Then Section 6 suddenly assumes everything has already been deployed. Took almost forty minutes to figure out what was missing. Error messages seen today: Connection timeout. Invalid token. File not found. Unexpected NULL value. Shopping list milk bread charger (USB-C?) HDMI cable AA batteries Don't forget birthday gift. Date... 18th? Need to verify. Sometimes projects look simple until implementation begins. One requirement becomes five, five become twenty. Documentation gets 

In [50]:
chunks[0] # third nearest neighbor

"Meeting started around 9:15 AM although a few people joined later. Main discussion was about the migration timeline. Nobody seemed completely certain whether Phase 2 should begin next Monday or after the quarterly review. Need confirmation from the infrastructure team. Reminder: Check backup logs. Verify user permissions. Send updated spreadsheet. Maybe ask finance if budget approval has actually been received? Yesterday felt unusually busy. Emails, calls, more emails... somewhere in between there was a request to generate three reports but only two were completed before lunch. The third one? Still pending because the source data looked incomplete. Strange thing was, the totals matched but individual records didn't. Reference IDs: INV-20481 REQ-9982 PO# 77124-B Status = Pending Weather forecast says 32°C tomorrow. Not sure that's accurate because it was supposed to rain today as well. Instead, sunshine all afternoon. By 5 PM the clouds finally appeared, then disappeared again within t